# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
import numpy as np
import pandas as pd
import pickle
import shap
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.preprocessing import FunctionTransformer


c:\Users\myche\.conda\envs\dsi_participant\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()

#fires_dt.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
# Ensure 'month' is categorical (it should already be, but just in case)
fires_dt['month'] = fires_dt['month'].astype('category')

fires_target = fires_dt["area"]

fires_variables = fires_dt.drop(columns = ["area","day"])

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    fires_variables, fires_target, test_size=0.2, random_state=42
)

# Ensure X_train and X_test remain DataFrames
X_train = pd.DataFrame(X_train, columns=fires_variables.columns)
X_test = pd.DataFrame(X_test, columns=fires_variables.columns)

# Ensure y_train and y_test remain Series
y_train = pd.Series(y_train, name=fires_target.name)
y_test = pd.Series(y_test, name=fires_target.name)

# Print to verify
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


X_train shape: (413, 11)
y_train shape: (413,)
X_test shape: (104, 11)
y_test shape: (104,)


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [4]:
numerical_variables = ["coord_x", "coord_y", "ffmc", "dmc", "dc", "isi", "temp", "rh", "wind", "rain"]
categorical_variables = ["month"]

preproc1 = ColumnTransformer([
    ('num_scaler', StandardScaler(), numerical_variables),
    ('cat_encoder', OneHotEncoder(handle_unknown="ignore"), categorical_variables)
])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [5]:
# Determine which variables should be transformed 
# based on the distribution of the data.

# Visual method of determining distribution
#for variable in numerical_variables:
#    plt.figure(figsize=(6, 4))
#    sns.histplot(fires_dt[variable], bins=30, kde=True)
#    plt.title(f"Distribution of {variable}")
#    plt.show()

# Determine skew and output it as a numerical number
for variable in numerical_variables:
    print(variable + ": " + str(fires_dt[variable].skew()))


# Define the variable transformations
log_transform = FunctionTransformer(np.log1p) # Log Transformation
power_transform = PowerTransformer(method="yeo-johnson") #yeo-johnson handles negative values

# Define the transformation methods based on the skewness of
# the variable.
# Log transformations will be applied to variables right-skewness with a abs
# value that does not include the range of values between 0 and 1
# Yeo-Transformation will be applied to left skewness with an abs value 
#that does not include the range of values between 0 and 1
log_transform_features = ["isi", "rain"]  # Apply log to right-skewed features
yeo_johnson_features = ["ffmc", "dc"]  # Apply Yeo-Johnson to left-skewed features


preproc2 = ColumnTransformer([
    ("num_log", log_transform, log_transform_features),
    ("num_yeo_johnson", power_transform,yeo_johnson_features),
    ("num_scaler", StandardScaler(), list(set(numerical_variables) - set(log_transform_features) - set(yeo_johnson_features))),
    ("cat_encoder", OneHotEncoder(handle_unknown="ignore"), categorical_variables)
])


coord_x: 0.036245821612869086
coord_y: 0.41729624593033865
ffmc: -6.575605977178827
dmc: 0.5474977944865835
dc: -1.1004451245649132
isi: 2.5363252664156875
temp: -0.331172237347285
rh: 0.8629040078552522
wind: 0.5710011270000588
rain: 19.816343982813166


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [6]:
# Pipeline A = preproc1 + baseline
baseline_model = LinearRegression()  # Updated baseline model

pipeline_a = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", baseline_model)
])

In [7]:
# Pipeline B = preproc2 + baseline
baseline_model = LinearRegression()  # Updated baseline model

pipeline_b = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", baseline_model)
])

In [8]:
# Pipeline C = preproc1 + advanced model
advanced_model = RandomForestRegressor(n_estimators=100, random_state=42)  # Advanced model

pipeline_c = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", advanced_model)
])
#print("RandomForestRegressor Parameters:", advanced_model.get_params().keys())

In [9]:
# Pipeline D = preproc2 + advanced model
advanced_model = RandomForestRegressor(n_estimators=100, random_state=42)  # Advanced model

pipeline_d = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", advanced_model)
])
#print("RandomForestRegressor Parameters:", advanced_model.get_params().keys())

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [10]:
# Define hyperparameter grid for Linear Regression
param_grid_a_b = {
    "regressor__fit_intercept": [True, False],
    #"regressor__positive": [True, False]
}

# Hyperparameter grid for Random Forest (Pipelines C & D remains the same)
param_grid_c_d = {
    "regressor__n_estimators": [50, 100, 200],        # Number of trees in the forest
    "regressor__max_depth": [10, 20, None],           # Maximum depth of trees
    "regressor__min_samples_split": [2, 5, 10],       # Minimum samples to split a node
    "regressor__min_samples_leaf": [1, 2, 4],         # Minimum samples required per leaf
    "regressor__max_features": ["sqrt", "log2", None] # Number of features considered per split
}


# Define pipelines and their respective hyperparameter grids
pipelines_params = [
    ("A", pipeline_a, param_grid_a_b),
    ("B", pipeline_b, param_grid_a_b),
    ("C", pipeline_c, param_grid_c_d),
    ("D", pipeline_d, param_grid_c_d)
]

print("param_grid_a_b:", type(param_grid_a_b), param_grid_a_b)
print("param_grid_c_d:", type(param_grid_c_d), param_grid_c_d)


param_grid_a_b: <class 'dict'> {'regressor__fit_intercept': [True, False]}
param_grid_c_d: <class 'dict'> {'regressor__n_estimators': [50, 100, 200], 'regressor__max_depth': [10, 20, None], 'regressor__min_samples_split': [2, 5, 10], 'regressor__min_samples_leaf': [1, 2, 4], 'regressor__max_features': ['sqrt', 'log2', None]}


In [11]:
print("param_grid_a_b keys:", list(param_grid_a_b.keys()))
print("param_grid_c_d keys:", list(param_grid_c_d.keys()))


best_models = {}

for name, pipeline, param_grid in pipelines_params:
    print(f"Running GridSearch for Pipeline {name}...")  # Debugging print
    
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2", n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_models[name] = grid_search.best_estimator_
    
    print(f"Pipeline {name}: Best Params = {grid_search.best_params_}, Best R² = {grid_search.best_score_:.4f}")



param_grid_a_b keys: ['regressor__fit_intercept']
param_grid_c_d keys: ['regressor__n_estimators', 'regressor__max_depth', 'regressor__min_samples_split', 'regressor__min_samples_leaf', 'regressor__max_features']
Running GridSearch for Pipeline A...
Pipeline A: Best Params = {'regressor__fit_intercept': False}, Best R² = -0.2340
Running GridSearch for Pipeline B...
Pipeline B: Best Params = {'regressor__fit_intercept': False}, Best R² = -0.1655
Running GridSearch for Pipeline C...
Pipeline C: Best Params = {'regressor__max_depth': 20, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}, Best R² = -0.2541
Running GridSearch for Pipeline D...
Pipeline D: Best Params = {'regressor__max_depth': 20, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}, Best R² = -0.2478


In [ ]:
# Dictionary to store best models
best_models = {}

# Perform GridSearch for each pipeline
for name, pipeline, param_grid in pipelines_params:
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2", n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_models[name] = grid_search.best_estimator_  # Store best model
    print(f"Pipeline {name}: Best Params = {grid_search.best_params_}, Best R² = {grid_search.best_score_:.4f}")

#Identify best-performing pipeline
best_pipeline_name = max(best_models, key=best_models.get)
best_pipeline = best_models[best_pipeline_name]
print(f"\n Best Model: Pipeline {best_pipeline_name} with Test R² = {best_models[best_pipeline_name]:.4f}")



Pipeline A: Best Params = {'regressor__fit_intercept': False}, Best R² = -0.2340
Pipeline B: Best Params = {'regressor__fit_intercept': False}, Best R² = -0.1655
Pipeline C: Best Params = {'regressor__max_depth': 20, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}, Best R² = -0.2541
Pipeline D: Best Params = {'regressor__max_depth': 20, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}, Best R² = -0.2478


TypeError: '>' not supported between instances of 'Pipeline' and 'Pipeline'

# Evaluate

+ Which model has the best performance? - Pipeline 

# Export

+ Save the best performing model to a pickle file.

In [13]:
# Save the best model
with open("best_model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print(f"✅ Best model saved as 'best_model.pkl'")

NameError: name 'best_pipeline' is not defined

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
# Explain model predictions using SHAP
explainer = shap.Explainer(best_pipeline.named_steps["regressor"], best_pipeline.named_steps["preprocessing"].transform(X_train))
shap_values = explainer(best_pipeline.named_steps["preprocessing"].transform(X_test))

# Select a sample observation
sample_index = 10  # Change this index to analyze different test samples
shap.waterfall_plot(shap_values[sample_index])

shap.summary_plot(shap_values, best_pipeline.named_steps["preprocessing"].transform(X_test))


*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.